# Module 085: Milestone Project - Personal Finance Database App

Phase 9: Databases & Web Apps | Duration: 3 hours | Milestone

## Setup: Models and Engine

In [ ]:
from datetime import datetime, date
from typing import List, Tuple, Dict, Optional

from sqlalchemy import (
    create_engine, Column, Integer, Float, String, DateTime, func,
)
from sqlalchemy.orm import declarative_base, sessionmaker, Session

Base = declarative_base()

class Transaction(Base):
    __tablename__ = 'transactions'
    id = Column(Integer, primary_key=True)
    date = Column(DateTime, nullable=False, default=datetime.now)
    amount = Column(Float, nullable=False)
    category = Column(String(100), nullable=False)
    description = Column(String(255), default='')

engine = create_engine('sqlite:///finance.db')
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session: Session = Session()
print('Database initialized')

## CRUD Operations

In [ ]:
# Create
t1: Transaction = Transaction(
    date=datetime(2024, 6, 1),
    amount=5000.0,
    category='Salary',
    description='June salary',
)
t2: Transaction = Transaction(
    date=datetime(2024, 6, 5),
    amount=-150.0,
    category='Food',
    description='Groceries',
)
session.add_all([t1, t2])
session.commit()
print('Sample transactions added')

In [ ]:
# Read
transactions: List[Transaction] = session.query(Transaction).order_by(Transaction.date.desc()).all()
for t in transactions:
    print(f'{t.id}: {t.date.date()} | ${t.amount:+.2f} | {t.category} | {t.description}')

In [ ]:
# Aggregation: spending by category
report: List[Tuple[str, float]] = (
    session.query(
        Transaction.category,
        func.sum(Transaction.amount).label('total'),
    )
    .filter(Transaction.amount > 0)
    .group_by(Transaction.category)
    .order_by(func.sum(Transaction.amount).desc())
    .all()
)
for cat, total in report:
    print(f'{cat}: ${total:.2f}')

In [ ]:
# Monthly summary
rows: List[Tuple[str, float, float]] = (
    session.query(
        func.strftime('%Y-%m', Transaction.date).label('month'),
        func.sum(Transaction.amount).filter(Transaction.amount > 0).label('income'),
        func.sum(Transaction.amount).filter(Transaction.amount < 0).label('expenses'),
    )
    .group_by('month')
    .order_by('month')
    .all()
)
for month, income, expenses in rows:
    income_val: float = income or 0.0
    expenses_val: float = abs(expenses) if expenses else 0.0
    print(f'{month}: Income=${income_val:.2f}, Expenses=${expenses_val:.2f}, Net=${income_val - expenses_val:.2f}')

## Delete and Cleanup

In [ ]:
# Delete
session.query(Transaction).filter_by(id=t2.id).delete()
session.commit()
remaining: int = session.query(Transaction).count()
print(f'Remaining transactions: {remaining}')

session.close()
print('Session closed')

## Next Steps

The complete project with CLI menu is in `project/solution/finance_app.py`. Open `project/starter_code.py` and implement the TODO functions, then build the interactive menu.